In [44]:
import os
from time import sleep
from PIL import Image
import requests
from io import BytesIO

from dotenv import load_dotenv
from huggingface_hub import login
from smolagents import CodeAgent, OpenAIServerModel, ActionStep, tool, DuckDuckGoSearchTool, GoogleSearchTool

import helium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains


login()
load_dotenv()


True

In [62]:
def initialize_driver():
    """Initialize the Selenium WebDriver."""
    chrome_options = webdriver.ChromeOptions()

    chrome_options.binary_location = "/usr/bin/chromium"

    chrome_options.add_argument("--force-device-scale-factor=1")
    chrome_options.add_argument("--window-size=1000,1350")
    chrome_options.add_argument("--disable-pdf-viewer")
    chrome_options.add_argument("--window-position=0,0")
    return helium.start_chrome(headless=False, options=chrome_options)

In [35]:
api_key = os.environ.get("OPEN_ROUTER_KEY", "")
if api_key == "":
    raise ValueError("api key not found")

open_router_base_url = os.environ.get("OPEN_ROUTER_BASE_URL", "")
if open_router_base_url == "":
    raise ValueError("open router url is required!")

serper_api_key = os.environ.get("SERPER_API_KEY", "")
if serper_api_key == "":
    raise ValueError("serper api key is required!")

In [ ]:
image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg", # Joker image
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg" # Joker image
]

images = []
for url in image_urls: 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    }
    response = requests.get(url=url, headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

In [59]:
@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
      Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(
        By.XPATH,
        f"//*[contains(text(), '{text}')]",
    )

    if nth_result < 1 or nth_result > len(elements):
        raise ValueError(
            f"Match {nth_result} not found "
            f"(only {len(elements)} matches found)"
        ) 

    elem = elements[nth_result-1]
    driver.execute_script(
        "arguments[0].scrollIntoView(true)",
        elem,
    )

    return f"Found {len(elements)} matches and focused on match {nth_result}."

@tool
def go_back() -> str:
    """ Navigate to the previous page."""
    driver = get_browser_driver()
    driver.back()
    return "Navigate to the previous page"

@tool
def close_popups() -> str:
    """Attempt to dismiss a popup by pressing Escape."""
    ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    return "Sent the Escape Key"

In [ ]:
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0) # Let Javascript animation happen before taking the screenshot

    driver = helium.get_driver()

    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps: # Remove previous screenshot from logs for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None

            # Capture the current browser screenshot
            png_bytes = driver.get_screenshot_as_png()
            image = Image.open(BytesIO(png_bytes))

            print(f"Captured a browser screenshot: {image.size} pixels")

            memory_step.observations_images = [image.copy()] # Create a copy to ensure it persists, important!

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    return


In [53]:
model = OpenAIServerModel(
    model_id="deepseek/deepseek-v4.1-flash",
    api_key=api_key,
    api_base=open_router_base_url, 
)

agent = CodeAgent(
    tools=[GoogleSearchTool(serper_api_key), go_back, close_popups, search_item_ctrl_f],
    model=model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=15,
    verbosity_level=2
)

In [54]:
helium_instructions = """
Use your web_search tool when you want to get Google search results.
Then you can use helium to access websites. Don't use helium for Google search, only for navigating websites!
Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
<code>
go_to('github.com/trending')
</code>

You can directly click clickable elements by inputting the text that appears on them.
<code>
click("Top products")
</code>

If it's a link:
<code>
click(Link("Top products"))
</code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
<code>
scroll_down(num_pixels=1200) # This will scroll one viewport down
</code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
<code>
close_popups()
</code>

You can use .exists() to check for the existence of an element. For example:
<code>
if Text('Accept cookies?').exists():
    click('I accept')
</code>

Proceed in several steps rather than trying to solve the task in one shot.
And at the end, only when you have your answer, return your final answer.
<code>
final_answer("YOUR_ANSWER_HERE")
</code>

If pages seem stuck on loading, you might have to wait, for instance `import time` and run `time.sleep(5.0)`. But don't overuse this!
To list elements on page, DO NOT try code-based element searches like 'contributors = find_all(S("ol > li"))': just look at the latest screenshot you have and read it visually, or use your tool search_item_ctrl_f.
Of course, you can act on buttons like a user would do when navigating.
After each code blob you write, you will be automatically provided with an updated screenshot of the browser and the current browser url.
But beware that the screenshot will only be taken at the end of the whole action, it won't see intermediate states.
Don't kill the browser.
When you have modals or cookie banners on screen, you should get rid of them before you can click anything else.
"""


task = """
I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.
Please search for images of Wonder Woman and generate a detailed visual description based on those images. Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can determine whether to grant her access to the event.
""" + helium_instructions


In [ ]:
global driver
driver = initialize_driver()
response = agent.run(task)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero  │
│ has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.   │
│ Please search for images of Wonder Woman and generate a detailed visual description based on those images.      │
│ Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can    │
│ determine whether to grant her access to the event.                                                             │
│                                                                                                                 │
│ Use your web_search tool when you want to get Google search results.                                            │
│ Then you can use helium to access websites. Don't use helium for Google search, only for navigating websites!   │
│ Don't bother about the helium driver, it's already managed.                                                     │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ <code>                                                                                                          │
│ go_to('github.com/trending')                                                                                    │
│ </code>                                                                                                         │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ <code>                                                                                                          │
│ click("Top products")                                                                                           │
│ </code>                                                                                                         │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ <code>                                                                                                          │
│ click(Link("Top products"))                                                                                     │
│ </code>                                                                                                         │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ <code>                                                                                                          │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ </code>                                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I'll help you verify the identity of this "Wonder Woman" guest. Let me start by searching for images of Wonder     
Woman and gathering information about her appearance.                                                              
                                                                                                                   
<code>                                                                                                             
results = web_search(query="Wonder Woman character images appearance")                                             
print(results)                                                                                                     
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search(query="Wonder Woman character images appearance")                                           
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [character.ai | AI Chat, Reimagined–Your Words. Your World.](https://character.ai/)

Chat with millions of AI Characters on the #1 AI chat app. Where will your next adventure take you? Get access 
to10M+ Characters Sign up in just ten seconds

1. [CHARACTER Definition & Meaning](https://www.merriam-webster.com/dictionary/character)
Date published: 7 days ago

The meaning of CHARACTER is a distinguishing attribute or feature : characteristic. This is a side of her character
[=personality] that few ...

2. [Character (Java Platform SE 8 )](https://docs.oracle.com/javase/8/docs/api/java/lang/Character.html)

Some characters look like pairs of Latin letters. For example, there is an uppercase letter that looks like "LJ" 
and has a corresponding lowercase letter ...

3. [Character.org: Home](https://character.org/)

Every year, Character.org celebrates National Schools of Character whose commitment to character development is 
evident throughout their school communities.

4. [Character | Apple Developer Documentation](https://developer.apple.com/documentation/swift/character)

Overview. The Character type represents a character made up of one or more Unicode scalar values, grouped by a 
Unicode boundary algorithm.

5. [Character requirements for visas - Immigration and 
citizenship](https://immi.homeaffairs.gov.au/help-support/meeting-our-requirements/character)

Understand character requirements for visas and citizenship applications, including assessments and obligations.

6. [The "Brain Dump" Method for Character Design](https://www.youtube.com/watch?v=oZfgTXDwuLI)
Date published: 9 months ago

Let's sketch a traveling merchant to practice character thumbnailing. We'll focus on generating unique silhouettes 
and rapid ideas.

7. [What Does “Character” Even Mean?](https://fullfocus.co/defining-character/)
Date published: Mar 27, 2018

Moral Virtues: Honesty, compassion, justice, gratitude, and forgiveness. Moral Vices: Dishonesty, cold-heartedness,
injustice, ungratefulness, ...

8. [Character Focus Initiative - National Association for 
...](https://www.nacacnet.org/character-focus-initiative/)

The Character Focus Initiative promotes the importance of character assessment in the college admission process. 
The initiative produces research-based ...

9. [Character Design Challenge!](https://www.facebook.com/groups/CharacterDesignChallenge/)

Once a month our members gather here to challenge each other in a contest in which they share their unique designs 
of characters, based on a common theme, along ...

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Let me try importing the helium functions directly.                                                                
                                                                                                                   
<code>                                                                                                             
from helium import *                                                                                               
go_to("https://en.wikipedia.org/wiki/Wonder_Woman")                                                                
print("Navigated")                                                                                                 
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from helium import *                                                                                             
  go_to("https://en.wikipedia.org/wiki/Wonder_Woman")                                                              
  print("Navigated")                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Navigated

Out: None

Captured a browser screenshot: (832, 445) pixels
Captured a browser screenshot: (832, 445) pixels


AttributeError: 'NoneType' object has no attribute 'encode'